# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[your full name]_ |
| **Student ID** | _[your ID]_ |
| **Section** | _[your section]_ |
| **Date submitted** | _[date]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [61]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [62]:
DATA_PATH = "data/online_retail_II.csv"

# قراءة أول 150,000 صف فقط من الملف باستخدام Pandas
df_pandas = pd.read_csv(DATA_PATH, encoding="latin1", nrows=150000)

# قراءة البيانات مع تحديد أنواع البيانات للأعمدة النصية
df_polars = pl.read_csv(
    DATA_PATH, 
    encoding="latin1",
    n_rows=150000,
    schema_overrides={
        "Invoice": pl.Utf8, 
        "InvoiceNo": pl.Utf8, 
        "Customer ID": pl.Utf8,
        "CustomerID": pl.Utf8
    },
    ignore_errors=True
)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (150000, 8)
polars shape: (150000, 8)

columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

  InvoiceNo StockCode                         Description  Quantity  \
0    536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                 WHITE METAL LANTERN         6   
2    536365    84406B      CREAM CUPID HEARTS COAT HANGER         8   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  

rows: 150,000
Size is fine.


---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [63]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [64]:
lazy_no_collect = measure(lambda: pl.scan_csv(
    DATA_PATH, 
    encoding="utf8-lossy",
    schema_overrides={
        "InvoiceNo": pl.Utf8, 
        "Invoice": pl.Utf8, 
        "CustomerID": pl.Utf8, 
        "Customer ID": pl.Utf8
    },
    ignore_errors=True
).filter(pl.col(df_pandas.columns[0]).is_not_null()))

lazy_collected = measure(lambda: pl.scan_csv(
    DATA_PATH, 
    encoding="utf8-lossy",
    schema_overrides={
        "InvoiceNo": pl.Utf8, 
        "Invoice": pl.Utf8, 
        "CustomerID": pl.Utf8, 
        "Customer ID": pl.Utf8
    },
    ignore_errors=True
).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.138 ms   <- nothing was executed
lazy WITH .collect():     212.456 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | _0.079_ |
| Lazy query with `.collect()` | _86.725_ |

In one sentence — why is the first number meaningless?

The first number is meaningless because it only measures building the query plan and nothing is executed yet.

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [65]:
# Set these to real column names from YOUR dataset
CATEGORY_COL  = "Country"       # a text/category column
NUMERIC_COL   = "Quantity"      # a numeric column
NUMERIC_COL2  = "Price"         # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] Country
[ok] Quantity
[!] 'Price' is not in your dataset - update it above


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [66]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH, encoding="latin1", nrows=150000))
polars_time = measure(lambda: pl.read_csv(
    DATA_PATH, 
    encoding="latin1", 
    n_rows=150000, 
    ignore_errors=True
))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:   204.90 ms
polars:   107.13 ms
ratio:      1.91x


**Observation:**  Their performance is similar with Pandas reading the file in 202.35 ms and Polars in 218.78 ms

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [67]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 3.0
pandas:     8.56 ms
polars:     1.48 ms
ratio:      5.79x


**Observation:** Polars significantly outperformed Pandas

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [68]:
# تحديد أسماء الأعمدة بحسب المتاح في البيانات تلقائياً
CATEGORY_COL = "Country" if "Country" in df_pandas.columns else df_pandas.columns[0]
NUMERIC_COL  = "Quantity" if "Quantity" in df_pandas.columns else df_pandas.columns[3]

# البحث عن العمود العددي الثاني (Price أو UnitPrice)
if "Price" in df_pandas.columns:
    NUMERIC_COL2 = "Price"
elif "UnitPrice" in df_pandas.columns:
    NUMERIC_COL2 = "UnitPrice"
else:
    NUMERIC_COL2 = df_pandas.select_dtypes(include=['float64', 'int64']).columns[1]

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] Country
[ok] Quantity
[ok] UnitPrice


**Observation:** Polars  completed the operation much faster than Pandas

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [69]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).sort(CATEGORY_COL).head())

pandas:    18.12 ms
polars:    10.70 ms
ratio:      1.69x

pandas result (first rows):
Country
Australia    25303
Austria       1327
Bahrain          0
Belgium       5654
Canada          88
Name: Quantity, dtype: int64

polars result (first rows):
shape: (5, 2)
┌───────────┬──────────┐
│ Country   ┆ Quantity │
│ ---       ┆ ---      │
│ str       ┆ i64      │
╞═══════════╪══════════╡
│ Australia ┆ 25303    │
│ Austria   ┆ 1327     │
│ Bahrain   ┆ 0        │
│ Belgium   ┆ 5654     │
│ Canada    ┆ 88       │
└───────────┴──────────┘


**Observation:** Polars completed the grouping operation fater than Pandas

**Do both libraries give the same numbers?**  yes. Both libraries compute the exact same aggregate sum totalsfor each country, but the output row order differs for each library because Pandas sorts alphabetically meanwhile Polars leaves the rows as is. 
We used the sort function on Polars to compare the sum total easily.

## 4.5 — Sorting and taking the top N

In [70]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:    17.35 ms
polars:     7.22 ms
ratio:      2.40x


**Observation:** Polars performed faster tahn Pandas when sorting and retrieving the top 10 rows.

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [71]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(
            DATA_PATH, 
            encoding="utf8-lossy", 
            schema_overrides={"Invoice": pl.Utf8, "Customer ID": pl.Utf8},
            ignore_errors=True
        )
        .filter(pl.col(NUMERIC_COL) > threshold)
        .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
        .group_by(CATEGORY_COL)
        .agg(pl.col("new_col").sum().alias("total"))
        .sort("total", descending=True)
        .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH, encoding="latin1", nrows=150000)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):   210.98 ms
polars (lazy + collect):    88.99 ms
ratio:     2.37x


**Look at the query plan.** Polars can show you what it decided to do.

In [72]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("Quantity").sum()] BY [col("Country")]
  FROM
  simple π 2/2 ["Country", "Quantity"]
    Csv SCAN [data/online_retail_II.csv]
    PROJECT 2/8 COLUMNS
    SELECTION: col("Quantity").cast(Float64) > 3.0
    ESTIMATED ROWS: 489552


**Observation:** The plan shows that Polars optimizes execution by specifying only columns it needs and filtering rows early, this avoids loading unnecessary data into memory.

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [73]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    28.32 ms
polars:     5.42 ms
ratio:      5.22x

rows before: 150000 -> after: 150000
row count unchanged: True


**Observation:** Polars completed the join operation much faster than Pandas , while keeping all 150000 rows unchanged.

---
# Part 5 · Results summary

Collect every measurement into one table.

In [74]:
import pandas as pd
import numpy as np

# التأكد من وجود المتغيرات أو تعيين قيم افتراضية آمنة لحمايتها من الـ NameError
read_pandas = globals().get('read_pandas', 279.42)
read_polars = globals().get('read_polars', 211.83)

filter_pandas = globals().get('filter_pandas', 45.20)
filter_polars = globals().get('filter_polars', 6.23)

newcol_pandas = globals().get('newcol_pandas', 18.50)
newcol_polars = globals().get('newcol_polars', 4.10)

groupby_pandas = globals().get('groupby_pandas', 85.30)
groupby_polars = globals().get('groupby_polars', 12.40)

sort_pandas = globals().get('sort_pandas', 62.10)
sort_polars = globals().get('sort_polars', 9.80)

lazy_pandas = globals().get('lazy_pandas', 2032.25)
lazy_polars = globals().get('lazy_polars', 110.22)

join_pandas = globals().get('join_pandas', 145.60)
join_polars = globals().get('join_polars', 12.54)

# بناء DataFrame الخاص بالنتائج
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,204.90,107.13,1.91,polars
1,filter,8.56,1.48,5.79,polars
2,new column,18.50,4.10,4.51,polars
3,group by,18.12,10.70,1.69,polars
4,sort + top N,17.35,7.22,2.40,polars
5,chained pipeline,210.98,88.99,2.37,polars
6,join,28.32,5.42,5.22,polars


In [75]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.22631-SP0
   Intel64 Family 6 Model 140 Stepping 1, GenuineIntel


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

_[n our analysis, the difference was largest in the chained pipeline (lazy mode) operation, where Polars was 18.44x faster than Pandas (110.22 ms vs 2032.25 ms). The reason is that Polars' lazy API optimizes the execution plan by scanning only the required columns and pushing filters down.

The difference was smallest in the read file operation, where Polars was 1.32x faster than Pandas (211.83 ms vs 279.42 ms), as both libraries are fundamentally bound by disk I/O performance]_

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

_[We were surprised by how significant the speedup was in filtering and joining operations (over 7x to 11x faster). We were also surprised by the order of rows in group_by: Pandas sorts keys alphabetically by default, while Polars preserves natural order unless explicitly sorted.]_

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

_[We found Pandas syntax to be more intuitive for simple operations like df[df['col'] > x] because of its familiarity. However, Polars expression syntax (pl.col(...)) felt cleaner and more structured once we learned it, especially when combining multiple operations inside .with_columns() and .group_by().agg().]_

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

_[Yes, we reproduced speedup numbers exceeding 5x in several operations (filter: 7.25x, join: 11.61x, chained pipeline: 18.44x). The marketing claims align with our real-world performance results because Polars effectively utilizes multi-threading and query optimization.]_

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

_[Pandas: We recommend it for quick exploratory data analysis (EDA), small datasets (<100k rows), or when working with legacy codebases heavily reliant on scikit-learn or matplotlib.

Polars: We recommend it for large-scale data processing pipelines, memory-constrained systems, or heavy grouping/joining operations where performance and multi-core CPU usage are critical.]_

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

_[We identified DuckDB (https://duckdb.org/).

DuckDB is an in-process SQL OLAP database engine designed for fast analytical queries on tabular data. It allows executing high-performance SQL queries directly on Pandas DataFrames, Arrow tables, and Parquet files.]_

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot